# 01 · PDF → Markdown Pipeline (DeepSeek-OCR-2)

**Hardware**: 🟡 NVIDIA GPU 16GB+ (depends on flash-attn — no local Mac support; use a cloud GPU, or jump to the general-VLM fallback in §4)

## What you will learn

1. The standard document-pipeline skeleton: **PDF → page images → OCR model → stitched Markdown**
2. DeepSeek-OCR's key knobs: how resolution tiers (`base_size`/`crop_mode`) trade quality against token cost
3. Why grounded output (with coordinates) is the key to anti-hallucination and traceability
4. Cheap quality checks: catching "confident fabrication" in OCR output

> Version note (2026-08): defer to the [DeepSeek-OCR-2 model card](https://huggingface.co/deepseek-ai/DeepSeek-OCR-2) for environment requirements (at release: torch 2.6 + flash-attn 2.7.3 + transformers 4.46).

In [ ]:
%pip install -q torch transformers tokenizers einops addict easydict pypdfium2 pillow
# On a GPU box, also: %pip install -q flash-attn --no-build-isolation

## 1. PDF → page images

OCR VLMs eat images, so step one is always rendering. **DPI is the first quality knob**: too low and small text smears; too high and you waste tokens. 150–200 DPI is the usual starting point.

In [ ]:
import pypdfium2 as pdfium
import requests, pathlib

# Example: an arXiv paper (Janus-Pro) as the test document — swap in your own PDF
pdf_path = pathlib.Path("sample.pdf")
if not pdf_path.exists():
    pdf_path.write_bytes(requests.get("https://arxiv.org/pdf/2501.17811", timeout=60).content)

pdf = pdfium.PdfDocument(pdf_path)
print(f"{len(pdf)} pages")

pages_dir = pathlib.Path("pages"); pages_dir.mkdir(exist_ok=True)
page_files = []
for i in range(min(3, len(pdf))):  # first 3 pages for the demo
    bitmap = pdf[i].render(scale=200 / 72)  # 200 DPI
    img = bitmap.to_pil()
    f = pages_dir / f"page_{i:03d}.png"
    img.save(f)
    page_files.append(f)
    print(f"{f}  {img.size}")

## 2. Load DeepSeek-OCR-2

The model ships its inference logic via `trust_remote_code` (the code lives in the model repo). Reading that remote code before running it is a good habit.

In [ ]:
import torch
from transformers import AutoModel, AutoTokenizer

MODEL_ID = "deepseek-ai/DeepSeek-OCR-2"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModel.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    use_safetensors=True,
    attn_implementation="flash_attention_2",
    torch_dtype=torch.bfloat16,
).eval().cuda()

## 3. Per-page OCR → stitched Markdown

Two things matter here:

- **`<|grounding|>`** in the prompt makes the output carry layout coordinates — every text block maps back to the source image, the basis of traceability
- **`base_size` / `crop_mode`** control the visual-token budget: cheaper tiers miss small print and dense tables — this is the "optical compression ratio" dial from the chapter's theory

In [ ]:
PROMPT = "<image>\n<|grounding|>Convert the document to markdown."

md_pages = []
for f in page_files:
    res = model.infer(
        tokenizer,
        prompt=PROMPT,
        image_file=str(f),
        output_path="ocr_out",
        base_size=1024, image_size=640, crop_mode=True,  # tier parameters: see model card
        save_results=True,
    )
    md_pages.append(res if isinstance(res, str) else open(f"ocr_out/{f.stem}.md").read())
    print(f"{f.name} done")

full_md = "\n\n---\n\n".join(md_pages)
pathlib.Path("output.md").write_text(full_md)
print(full_md[:1500])

## 4. Fallback: a general VLM when you have no NVIDIA GPU

Lower ceiling, no grounding — but runs on a Mac or through APIs. It also doubles as the comparison baseline (this is the seed of `02_vlm_vs_ocr_model` in the notebooks index).

In [ ]:
# Transcribe page by page with chapter 01's Qwen3-VL or any closed API:
# prompt = ("Convert this document page to complete Markdown. Use Markdown tables for tables. "
#           "Mark unreadable characters with \u25a1. Guessing is forbidden.")
# Reuse chat() from 01-vlm/notebooks/01_qwen3vl_local.ipynb — omitted here.

## 5. Quality check: catching confident fabrication

End-to-end OCR's most dangerous failure mode isn't missing text — it's **rendering unreadable content as fluent inventions**. Two cheap health checks:

1. **Digit audit**: regex out all numbers and spot-check a sample against the source — digits are the hallucination hotspot with the worst consequences
2. **Round-trip audit**: render the generated Markdown back to an image and have another VLM diff the two (expensive but fully automatic)

In [ ]:
import re
numbers = re.findall(r"\d+\.?\d*", full_md)
print(f"extracted {len(numbers)} numbers, sample: {numbers[:20]}")
print("Manually verify 10 of them against the source PDF — the cheapest acceptance test there is.")

## Exercises

1. Rerun one page across `base_size` 768/1024/1280 and compare formula and footnote fidelity — plot the quality-vs-token curve.
2. Try a photo of a tilted receipt (not a scan) and see how the pipeline handles real-world input.
3. Score your pipeline on a sample of [OmniDocBench](https://github.com/opendatalab/OmniDocBench).
4. Chunk the output Markdown into a vector store — you have just built the front half of chapter 08's multimodal RAG.